1. Is our Gold file's Answerable rate also low?
That's the number that matters. Run:

In [3]:
import pandas as pd
import os

# The path to the data directory
DATA_DIR = "/vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data"
parquet_path = os.path.join(DATA_DIR, "RVS_MASTER_GOLD_HYDRATED.parquet")


# Verify the file exists before loading to avoid FileNotFoundError
if os.path.exists(parquet_path):
    print(f"✅ Found {parquet_path}, loading...")
else:
    print(f"❌ Could not find {parquet_path}. Check your path!")

✅ Found /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data/RVS_MASTER_GOLD_HYDRATED.parquet, loading...


In [4]:
import pandas as pd
import os

# 1. Define the absolute path clearly
full_path = "/vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/data/RVS_MASTER_GOLD_HYDRATED.parquet"

# 2. Load using the full path variable
if os.path.exists(full_path):
    df = pd.read_parquet(full_path)
    print(f"✅ Loaded {len(df)} rows.")
    
    # 3. Run our analysis
    stats = df[df['city'] == 'pittsburgh']['oracle_label'].value_counts(normalize=True)
    print("\n📊 Pittsburgh Label Distribution:")
    print(stats)
else:
    print(f"❌ Error: File not found at {full_path}")

✅ Loaded 9301 rows.

📊 Pittsburgh Label Distribution:
oracle_label
Answerable       0.760508
Contradictory    0.173998
Ambiguous        0.065494
Name: proportion, dtype: float64


2. Are Contradictory samples concentrated in specific categories?

In [5]:
import pandas as pd
import os

# Define the base project path
BASE_DIR = "/vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning"

# 1. Load the new Silver Standard we just generated
silver_path = os.path.join(BASE_DIR, "data/pittsburgh/pittsburgh_silver_standard.parquet")
silver = pd.read_parquet(silver_path)

# 2. Run your analysis on the "failures"
print("🔍 Distribution of Categories for Contradictory Samples:")
print(silver[silver['oracle_label'] == 'Contradictory']['extracted_category'].value_counts())

# 3. Quick check: Are there many 'None' or 'unmapped' categories?
total_contradictory = len(silver[silver['oracle_label'] == 'Contradictory'])
missing_cat = silver[silver['oracle_label'] == 'Contradictory']['extracted_category'].isna().sum()
print(f"\n⚠️ Out of {total_contradictory} contradictions, {missing_cat} had NO category extracted.")

🔍 Distribution of Categories for Contradictory Samples:
extracted_category
UNKNOWN    504
ROAD        31
Name: count, dtype: int64

⚠️ Out of 535 contradictions, 0 had NO category extracted.


Meaning: The old pipeline had 76% Answerable and the new one has 30%.

The regression is entirely in our recent (remotely done) bug fixes and optimizations, not in the data.

The Smoking Gun
Looking at the old Contradictory breakdown:

506 out of 578 contradictions were UNKNOWN category — meaning the solver couldn't extract a category, so it couldn't find candidates, so it labeled Contradictory.

That was always the dominant failure mode. Our new pipeline has 578 total Contradictory samples — roughly the same absolute number as the old pipeline's UNKNOWN-category contradictions alone. So somehow the recent fixes caused nearly everything that was Answerable to become Contradictory.

Diagnostic results from `labeling_284223_1.out`:
⚠️ Direction filter dropped 5/5 candidates due to unresolved node IDs
⚠️ Direction filter dropped 239/284 candidates due to unresolved node IDs
⚠️ Direction filter dropped 39/42 candidates due to unresolved node IDs
⚠️ Direction filter dropped 39/42 candidates due to unresolved node IDs
⚠️ Direction filter dropped 17/17 candidates due to unresolved node IDs
⚠️ Direction filter dropped 15/15 candidates due to unresolved node IDs

Reveal that since the diagnostic is firing constantly — candidates are being found but then almost all are dropped because their node IDs can't be resolved. The integer cast fix is what's needed.

In [6]:
import pickle
with open("../data/pittsburgh/pittsburgh_graph.gpickle", "rb") as f:
    G = pickle.load(f)

sample_nodes = list(G.nodes())[:5]
print(type(sample_nodes[0]), sample_nodes)

/tmp/ipykernel_3173135/423371716.py:3: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


<class 'str'> ['1#34184938', '#34184938', '1#105987017', '#105987017', '1#153846392']


In [7]:
import pickle
with open("../data/pittsburgh/pittsburgh_graph.gpickle", "rb") as f:
    G = pickle.load(f)

# Check if these specific candidates exist
test_ids = ['1#4970784495', '1#506804360', '1#5258254123', '1#543922276']
for nid in test_ids:
    print(f"{nid}: {'EXISTS' if nid in G.nodes else 'MISSING'}")

/tmp/ipykernel_3173135/535008901.py:3: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  G = pickle.load(f)


1#4970784495: EXISTS
1#506804360: EXISTS
1#5258254123: EXISTS
1#543922276: EXISTS


In [9]:
print(poi.columns.tolist())
print(poi.head(2))

['unique_id', 'osmid', 'element_type', 'highway', 'geometry', 'ref', 'crossing', 'barrier', 'railway', 'created_by', 'name', 'old_ref', 'place', 'name:en', 'name:he', 'name:oc', 'name:pdc', 'name:ru', 'population', 'short_name', 'source:name:oc', 'wikidata', 'wikipedia', 'network', 'public_transport', 'railway:ref', 'train', 'operator', 'bicycle', 'bus', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'source', 'unsigned_ref', 'foot', 'motor_vehicle', 'odbl', 'odbl:note', 'direction', 'tactile_paving', 'note', 'light_rail', 'local_ref', 'network:short', 'network:wikidata', 'bridge:support', 'destination', 'fixme', 'old_name', 'access', 'ele', 'gnis:Class', 'gnis:County', 'gnis:County_num', 'gnis:ST_alpha', 'gnis:ST_num', 'gnis:id', 'import_uuid', 'is_in', 'horse', 'wheelchair', 'shelter', 'natural', 'amenity', 'removed:highway', 'entrance', 'level', 'building', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:state', 'addr:street', 'description', 'brand', 'brand:w

In [10]:
import pandas as pd

# 1. Load the POI data
poi = pd.read_pickle("../data/pittsburgh/pittsburgh_poi.pkl")

# 2. Define the prefix from config
prefix = "1#"

def clean_and_check(raw_id):
    # Extract the numeric part (e.g., 'node/34184938' -> '34184938')
    numeric_id = str(raw_id).split('/')[-1]
    
    # Check 1: Numeric String (e.g., "34184938")
    if numeric_id in G:
        return True
    # Check 2: Prefixed String (e.g., "1#34184938")
    if f"{prefix}{numeric_id}" in G:
        return True
    # Check 3: Raw ID as is (just in case)
    if str(raw_id) in G:
        return True
        
    return False

# 3. Apply the check
poi['in_graph'] = poi['unique_id'].apply(clean_and_check)

# 4. Results
print(f"📊 Pittsburgh POI-to-Graph Coverage:")
print(poi['in_graph'].value_counts())
print(f"\nTotal Coverage Score: {poi['in_graph'].mean():.1%}")

# 5. Debugging: If coverage is still low, see what the IDs look like
if poi['in_graph'].mean() < 1.0:
    print("\n🧐 First 5 'Missing' IDs (Cleaned):")
    missing = poi[poi['in_graph'] == False]['unique_id'].head()
    for m in missing:
        print(f"Original: {m} | Cleaned: {m.split('/')[-1]} | Prefixed: {prefix}{m.split('/')[-1]}")
    
    print("\n🔍 Example nodes actually in G:")
    print(list(G.nodes())[:5])

/vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/anaconda3/envs/nlp_spatial/lib/python3.10/pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


📊 Pittsburgh POI-to-Graph Coverage:
in_graph
True     4496
False     502
Name: count, dtype: int64

Total Coverage Score: 90.0%

🧐 First 5 'Missing' IDs (Cleaned):
Original: node/281362013 | Cleaned: 281362013 | Prefixed: 1#281362013
Original: node/357375848 | Cleaned: 357375848 | Prefixed: 1#357375848
Original: node/357376642 | Cleaned: 357376642 | Prefixed: 1#357376642
Original: node/357377282 | Cleaned: 357377282 | Prefixed: 1#357377282
Original: node/357379242 | Cleaned: 357379242 | Prefixed: 1#357379242

🔍 Example nodes actually in G:
['1#34184938', '#34184938', '1#105987017', '#105987017', '1#153846392']
